# Evaluation Metrics - Image Classification
Master Data & IA, INPT 2026

Dataset: scikit-learn `digits` (1797 images 8x8, 10 classes). Model: Logistic Regression on pixels.
Run all cells with **Runtime > Run all**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, auc, average_precision_score,
    classification_report, cohen_kappa_score, confusion_matrix, f1_score,
    log_loss, matthews_corrcoef, mean_absolute_error, mean_squared_error,
    precision_recall_curve, precision_score, r2_score, recall_score,
    roc_curve, top_k_accuracy_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, label_binarize

SEED = 42

## 1. Data and training

In [ ]:
def make_model():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

digits = load_digits()
X, y = digits.data, digits.target
classes = np.unique(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
model = make_model().fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 2. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=classes).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion matrix (test set)")
plt.show()

## 3. Accuracy, Precision, Recall, F1

In [ ]:
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
for avg in ("macro", "micro", "weighted"):
    p = precision_score(y_test, y_pred, average=avg)
    r = recall_score(y_test, y_pred, average=avg)
    f = f1_score(y_test, y_pred, average=avg)
    print(f"{avg:>8}  precision={p:.4f}  recall={r:.4f}  f1={f:.4f}")
print()
print(classification_report(y_test, y_pred, digits=4))

## 4. ROC curves and AUC (one-vs-rest)

In [ ]:
y_test_bin = label_binarize(y_test, classes=classes)
fig, ax = plt.subplots(figsize=(7, 6))
aucs = []
for c in classes:
    fpr, tpr, _ = roc_curve(y_test_bin[:, c], y_proba[:, c])
    aucs.append(auc(fpr, tpr))
    ax.plot(fpr, tpr, lw=1, label=f"class {c} (AUC={aucs[-1]:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="random")
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC curves (one-vs-rest)")
ax.legend(fontsize=7, loc="lower right")
plt.show()
print(f"Macro AUC (OvR): {np.mean(aucs):.4f}")

## 5. Precision-Recall curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
aps = []
for c in classes:
    prec, rec, _ = precision_recall_curve(y_test_bin[:, c], y_proba[:, c])
    aps.append(average_precision_score(y_test_bin[:, c], y_proba[:, c]))
    ax.plot(rec, prec, lw=1, label=f"class {c} (AP={aps[-1]:.3f})")
ax.set(xlabel="Recall", ylabel="Precision", title="Precision-Recall curves")
ax.legend(fontsize=7, loc="lower left")
plt.show()
print(f"Mean Average Precision: {np.mean(aps):.4f}")

## 6. Top-k accuracy, Log loss, Kappa, MCC

In [ ]:
for k in (1, 3, 5):
    print(f"Top-{k} accuracy : {top_k_accuracy_score(y_test, y_proba, k=k, labels=classes):.4f}")
print(f"Log loss       : {log_loss(y_test, y_proba):.4f}")
print(f"Cohen's Kappa  : {cohen_kappa_score(y_test, y_pred):.4f}")
print(f"MCC            : {matthews_corrcoef(y_test, y_pred):.4f}")

## 7. Stratified k-fold cross-validation

In [ ]:
K = 5
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
fold_acc, fold_f1 = [], []
for i, (tr, va) in enumerate(skf.split(X, y), start=1):
    m = make_model().fit(X[tr], y[tr])
    pred = m.predict(X[va])
    fold_acc.append(accuracy_score(y[va], pred))
    fold_f1.append(f1_score(y[va], pred, average="macro"))
    print(f"Fold {i}: accuracy={fold_acc[-1]:.4f}  macro-F1={fold_f1[-1]:.4f}")
print(f"\nAccuracy : {np.mean(fold_acc):.4f} +/- {np.std(fold_acc):.4f}")
print(f"Macro-F1 : {np.mean(fold_f1):.4f} +/- {np.std(fold_f1):.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(1, K + 1), fold_acc, color="#1f77b4")
ax.axhline(np.mean(fold_acc), color="red", ls="--", label=f"mean={np.mean(fold_acc):.3f}")
ax.set(xlabel="Fold", ylabel="Accuracy", ylim=(0.9, 1.0), title=f"{K}-fold accuracy")
ax.legend()
plt.show()

## 8. Regression metrics (contrast only, not for classification)

In [ ]:
y_true_reg = np.array([3.0, -0.5, 2.0, 7.0])
y_pred_reg = np.array([2.5, 0.0, 2.0, 8.0])
mse = mean_squared_error(y_true_reg, y_pred_reg)
print(f"MAE  : {mean_absolute_error(y_true_reg, y_pred_reg):.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {np.sqrt(mse):.4f}")
print(f"R2   : {r2_score(y_true_reg, y_pred_reg):.4f}")